In [1]:
import os
import requests
from urllib.parse import urlencode
from dotenv import load_dotenv
import msal

In [2]:
load_dotenv()

TENANT_ID = os.environ["TENANT_ID"]
CLIENT_ID = os.environ["CLIENT_ID"]
CLIENT_SECRET = os.environ["CLIENT_SECRET"]
SITE_ID = os.environ["SITE_ID"]
LIST_ID = os.environ["LIST_ID"]
SENDER_UPN = os.environ["SENDER_UPN"]

AUTHORITY = f"https://login.microsoftonline.com/{TENANT_ID}"
SCOPE = ["https://graph.microsoft.com/.default"]
GRAPH = "https://graph.microsoft.com/v1.0"

# Adjust these to your actual SharePoint internal column names.
# Display names like "Status" and "NJ" may have different internal names (e.g., Status, NJ).
STATUS_FIELD = "Status"   # internal name for "status" column
EMAIL_FIELD = "NJ"        # internal name for "NJ" (email) column
NOTIFIED_FIELD = "Notified"  # boolean yes/no column you added

def get_token():
    app = msal.ConfidentialClientApplication(
        client_id=CLIENT_ID,
        client_credential=CLIENT_SECRET,
        authority=AUTHORITY
    )
    result = app.acquire_token_silent(SCOPE, account=None)
    if not result:
        result = app.acquire_token_for_client(scopes=SCOPE)
    if "access_token" not in result:
        raise RuntimeError(f"Token error: {result}")
    return result["access_token"]

def graph_get(url, token):
    r = requests.get(url, headers={"Authorization": f"Bearer {token}"})
    r.raise_for_status()
    return r.json()

def graph_patch(url, token, json):
    r = requests.patch(url, json=json, headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    })
    r.raise_for_status()
    return r.json() if r.text else {}

def graph_post(url, token, json):
    r = requests.post(url, json=json, headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    })
    # sendMail returns 202 with empty body
    if r.status_code not in (200, 201, 202, 204):
        raise RuntimeError(f"POST {url} failed: {r.status_code} {r.text}")
    return r.json() if r.text else {}

def list_items_to_notify(token):
    """
    Pull items where:
      - Status == 'YES'
      - Notified != true (either false or missing)
    We select fields to keep payload small.
    """
    base = f"{GRAPH}/sites/{SITE_ID}/lists/{LIST_ID}/items"
    # Graph filter across list fields uses fields/<InternalName>
    params = {
        # filter for Status == 'YES' and (Notified ne true or null)
        "$filter": f"(fields/{STATUS_FIELD} eq 'YES') and (fields/{NOTIFIED_FIELD} ne true)",
        "$select": "id,fields",
        # Expand fields to read actual column values
        "$expand": "fields($select=Id,{0},{1},{2})".format(STATUS_FIELD, EMAIL_FIELD, NOTIFIED_FIELD)
    }
    url = f"{base}?{urlencode(params)}"

    items = []
    while url:
        data = graph_get(url, token)
        items.extend(data.get("value", []))
        url = data.get("@odata.nextLink")

    return items

def send_mail(token, to_email, subject, html_body):
    url = f"{GRAPH}/users/{SENDER_UPN}/sendMail"
    payload = {
        "message": {
            "subject": subject,
            "body": {"contentType": "HTML", "content": html_body},
            "toRecipients": [{"emailAddress": {"address": to_email}}],
        },
        "saveToSentItems": True
    }
    graph_post(url, token, payload)

def mark_notified(token, item_id):
    """
    PATCH the item's fields to set Notified = true
    """
    url = f"{GRAPH}/sites/{SITE_ID}/lists/{LIST_ID}/items/{item_id}/fields"
    graph_patch(url, token, {NOTIFIED_FIELD: True})

def main():
    token = get_token()
    items = list_items_to_notify(token)

    for it in items:
        fields = it.get("fields", {})
        to_addr = fields.get(EMAIL_FIELD)
        status_val = fields.get(STATUS_FIELD)

        if not to_addr:
            # Skip rows without NJ email
            continue

        subject = "Notification: Status is YES"
        body = f"""
            <p>Hello,</p>
            <p>This is an automated notification. The SharePoint item with ID <b>{fields.get('Id')}</b>
            has <b>Status = {status_val}</b>.</p>
            <p>Thank you.</p>
        """

        # 1) Send the email
        send_mail(token, to_addr, subject, body)

        # 2) Mark as notified to avoid duplicates
        mark_notified(token, it["id"])

    print(f"Processed {len(items)} item(s).")

if __name__ == "__main__":
    main()

ValueError: Unable to get authority configuration for https://login.microsoftonline.com/xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx. Authority would typically be in a format of https://login.microsoftonline.com/your_tenant or https://tenant_name.ciamlogin.com or https://tenant_name.b2clogin.com/tenant.onmicrosoft.com/policy.  Also please double check your tenant name or GUID is correct.